# VascuMap Batch Pipeline

Two workflow modes:

1. **GUI Curation** — Launch `CurationApp` to review & curate device ROI + organoid mask
   for every image in a napari viewer, then run the full pipeline unattended.
2. **Headless Automatic** — Skip the GUI and run fully automatic device segmentation +
   pipeline on every image.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

from liffile import LifFile

from vascumap import VascuMap
from gui_region_detection import CurationApp
from models import Pix2Pix, load_segmentation_model

W0416 15:27:58.735000 609744 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Helper functions ───────────────────────────────────────────────────────────

def discover_jobs(source_dir, force_mask=None, require_merged=True):
    """Find .lif/.tif/.tiff files and return (source, files, jobs) list."""
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")

    image_files = sorted(
        p for p in source.iterdir()
        if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
    )

    def _mask_mode(p, img_name=""):
        if force_mask is not None:
            return force_mask
        name = (p.name + " " + img_name).lower()
        if "marina" in name and "bead" in name:
            return "light"
        return "dark" if "marina" in name else False

    jobs = []
    for p in image_files:
        if p.suffix.lower() == ".lif":
            try:
                with LifFile(p) as lif:
                    for idx, img in enumerate(lif.images):
                        name = getattr(img, "name", "")
                        if require_merged and "merged" not in name.lower():
                            continue
                        jobs.append((p, idx, _mask_mode(p, name), name))
            except Exception as exc:
                print(f"[SKIP] {p.name}: {exc}")
        else:
            if require_merged and "merged" not in p.name.lower():
                continue
            jobs.append((p, 0, _mask_mode(p), p.stem))

    return source, image_files, jobs


def get_perfect_image_names(perfect_dir):
    """Return a set of image names that have already been perfectly segmented."""
    perfect = Path(perfect_dir)
    if not perfect.is_dir():
        print(f"[WARN] Perfect directory not found: {perfect}")
        return set()
    names = {d.name for d in perfect.iterdir() if d.is_dir()}
    print(f"Found {len(names)} perfect images in {perfect}")
    return names


def run_batch_from_curation(curated_jobs, output_base, save_all_interim=False,
                            model_p2p=None, model_unet=None, skip_names=None):
    """Run the full pipeline on curated jobs from CurationApp.

    Parameters
    ----------
    curated_jobs : list[CuratedJob]
        Jobs returned by ``CurationApp.run()`` — only those with
        status ``"curated"`` are processed.
    output_base : str | Path
        Root output directory.
    """
    results = []
    if skip_names is None:
        skip_names = set()

    Path(output_base).mkdir(parents=True, exist_ok=True)
    processable = [j for j in curated_jobs if j.status == "curated"
                   and hasattr(j, '_finalised_outputs') and j._finalised_outputs is not None]

    for i, job in enumerate(processable, 1):
        name = job.image_name
        if name in skip_names:
            print(f"\n[{i}/{len(processable)}] {name}  — SKIP (already perfect)")
            results.append((name, "SKIP_PERFECT", ""))
            continue

        print(f"\n{'='*70}\n[{i}/{len(processable)}] {name}\n{'='*70}")

        try:
            vm = VascuMap(
                curated_outputs=job._finalised_outputs,
                model_p2p=model_p2p,
                model_unet=model_unet,
            )
            out_dir = Path(output_base) / vm.image_name
            print(f"  Output → {out_dir}")
            vm.pipeline(output_dir=out_dir, save_all_interim=save_all_interim)
            results.append((vm.image_name, "OK", ""))
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((name, "FAILED", str(exc)))

    # Summary
    n_ok = sum(1 for _, s, _ in results if s == "OK")
    n_skip = sum(1 for _, s, _ in results if s == "SKIP_PERFECT")
    n_skipped_curation = sum(1 for j in curated_jobs if j.status == "skip")
    print(f"\n{'='*70}\nBatch complete: {n_ok}/{len(results)} succeeded, "
          f"{n_skip} skipped (perfect), {n_skipped_curation} skipped (curation)")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")

    return results


def run_batch(jobs, output_base, device_width_um, brightfield_channel=0,
              save_all_interim=False, model_p2p=None, model_unet=None,
              start_index=1, skip_names=None):
    """Run all jobs in headless automatic mode (no GUI curation)."""
    results = []
    if skip_names is None:
        skip_names = set()

    Path(output_base).mkdir(parents=True, exist_ok=True)
    failure_diag_dir = Path(output_base) / "FAILED_diagnostics"

    for i, (path, idx, mask_flag, img_name) in enumerate(jobs, start_index):
        tag = f" (LIF #{idx}: {img_name})" if path.suffix.lower() == ".lif" else ""

        if path.suffix.lower() == ".lif":
            safe_name = img_name.replace("/", "_").replace("\\", "_")
            expected_name = f"{path.stem}_{safe_name}_img{idx}"
        else:
            expected_name = path.stem

        if expected_name in skip_names:
            print(f"\n[{i}/{len(jobs)}] {path.name}{tag}  — SKIP (already perfect)")
            results.append((expected_name, "SKIP_PERFECT", ""))
            continue

        print(f"\n{'='*70}\n[{i}/{len(jobs)}] {path.name}{tag}  mask={mask_flag}\n{'='*70}")

        try:
            vm = VascuMap(
                image_source_path=str(path),
                image_index=idx,
                device_width_um=device_width_um,
                mask_central_region=mask_flag,
                brightfield_channel=brightfield_channel,
                model_p2p=model_p2p,
                model_unet=model_unet,
                failure_output_dir=str(failure_diag_dir),
            )
            vm.image_name = expected_name
            out_dir = Path(output_base) / vm.image_name
            print(f"  Output → {out_dir}")
            vm.pipeline(output_dir=out_dir, save_all_interim=save_all_interim)
            results.append((vm.image_name, "OK", ""))
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((path.name + tag, "FAILED", str(exc)))

    # Summary
    n_ok = sum(1 for _, s, _ in results if s == "OK")
    n_skip = sum(1 for _, s, _ in results if s == "SKIP_PERFECT")
    print(f"\n{'='*70}\nBatch complete: {n_ok}/{len(results)} succeeded, {n_skip} skipped (perfect)")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")

    return results

In [3]:
shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)


In [4]:
# ── Configuration (edit here) ──────────────────────────────────────────────────
source_dir  = r"Z:\Bel\Dorota_Vascumap_Check"
output_base = r"z:\Bel\test_delete"

use_gui               = False    # True = napari curation, False = fully automatic
device_width_um       = 35.0
brightfield_channel   = 0
force_mask            = None     # "dark" / "light" / False / None (auto)
require_merged        = False
save_all_interim      = False

In [5]:
# ── Discover jobs + run ────────────────────────────────────────────────────────
source, image_files, jobs = discover_jobs(source_dir, force_mask, require_merged)

print(f"Source: {source}\nFiles: {len(image_files)}  |  Jobs: {len(jobs)}")
for i, (p, idx, mask, img_name) in enumerate(jobs, 1):
    tag = f" (LIF image {idx}: '{img_name}')" if p.suffix.lower() == ".lif" else ""
    print(f"  {i}. {p.name}{tag}  mask={mask}")

# Skip images already categorised as perfect
perfect_dir = r"z:\Bel\test_delete"
perfect_names = get_perfect_image_names(perfect_dir)

if use_gui:
    app = CurationApp(source_dir, device_width_um=device_width_um)
    curated_jobs = app.run()
    run_batch_from_curation(curated_jobs, output_base, save_all_interim,
                            shared_model_p2p, shared_model_unet, skip_names=perfect_names)
else:
    run_batch(jobs, output_base, device_width_um, brightfield_channel,
              save_all_interim, shared_model_p2p, shared_model_unet,
              skip_names=perfect_names)

Source: Z:\Bel\Dorota_Vascumap_Check
Files: 1  |  Jobs: 1
  1. DZ_Permeability_Vascumap_check.lif (LIF image 0: 'P 1')  mask=False
Found 4 perfect images in z:\Bel\test_delete

[1/1] DZ_Permeability_Vascumap_check.lif (LIF #0: P 1)  mask=False
  [LIF] Raw array shape: (3, 2, 30, 512, 512)  dtype=uint8
  Failure log → z:\Bel\test_delete\FAILED_diagnostics\P 1_FAILED.txt
  [SKIP] [WARN] Draw or adjust geometry first.

Batch complete: 0/1 succeeded, 0 skipped (perfect)
  FAILED: DZ_Permeability_Vascumap_check.lif (LIF #0: P 1): [WARN] Draw or adjust geometry first.
